# Pawpularity Score Prediction — Demo Notebook

This notebook lets you:
1. Set up the environment
2. Load the pretrained baseline model
3. Run inference on sample images
4. Visualise Grad-CAM heatmaps
5. Reproduce the key results table

**Prerequisites:** Complete the setup steps in `README.md` (conda env + data download) before running this notebook.

## 0. Imports & Setup

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on the path so src.* imports work
REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

from src.model import PawpularityModel
from src.dataset import TABULAR_COLS, IMAGENET_MEAN, IMAGENET_STD, get_dataloaders
from src.utils import get_device, rmse, set_seed

set_seed(42)
device = get_device()
print(f"Device: {device}")
print(f"Repo root: {REPO_ROOT}")

## 1. Load the Pretrained Baseline Model

The checkpoint at `checkpoints/fusion_concat.pt` is the best model from full training (val RMSE 20.53).  
If you haven't trained yet, run `python -m src.train --config configs/default.yaml --debug_mode false --epochs 25` first.

In [ ]:
CKPT_PATH = REPO_ROOT / "checkpoints" / "fusion_concat.pt"

if not CKPT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found at {CKPT_PATH}.\n"
        "Run: python -m src.train --config configs/default.yaml --debug_mode false --epochs 25"
    )

ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
cfg = ck["cfg"]

model = PawpularityModel(
    fusion_mode=cfg.get("fusion_mode", "fusion_concat"),
    freeze_backbone=cfg["freeze_backbone"],
    tabular_input_dim=cfg["tabular_input_dim"],
    tabular_hidden_dim=cfg["tabular_hidden_dim"],
    tabular_output_dim=cfg["tabular_output_dim"],
    fusion_hidden_dim=cfg["fusion_hidden_dim"],
    dropout=cfg["dropout"],
)

# Remap old checkpoint keys if needed
state = {k.replace("fusion_head.", "head."): v for k, v in ck["model_state_dict"].items()}
model.load_state_dict(state, strict=False)
model.to(device).eval()

print(f"Loaded checkpoint: epoch={ck['epoch']}, val_rmse={ck['val_rmse']:.4f}")
print(f"Fusion mode: {cfg['fusion_mode']}")

## 2. Run Inference on Sample Images

We use the 100-image debug subset (`data/debug/`) so this works without the full dataset downloaded.

In [ ]:
DEBUG_CSV = REPO_ROOT / "data" / "debug" / "train.csv"
DEBUG_IMG_DIR = REPO_ROOT / "data" / "debug" / "train"

if not DEBUG_CSV.exists():
    raise FileNotFoundError(
        f"Debug data not found at {DEBUG_CSV}.\n"
        "Run: python scripts/download_data.py && python scripts/resize_images.py"
    )

df = pd.read_csv(DEBUG_CSV)
print(f"Debug dataset: {len(df)} samples")
print(f"Columns: {list(df.columns)}")
df.head(3)

In [ ]:
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def predict_sample(row):
    """Run inference on a single DataFrame row. Returns predicted score."""
    img_path = DEBUG_IMG_DIR / (row["Id"] + ".jpg")
    img = Image.open(img_path).convert("RGB")
    img_tensor = val_transform(img).unsqueeze(0).to(device)
    tab_tensor = torch.tensor(
        row[TABULAR_COLS].values.astype(np.float32)
    ).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(img_tensor, tab_tensor).item()
    return pred

# Run on first 8 samples
samples = df.sample(8, random_state=42).reset_index(drop=True)
samples["predicted"] = samples.apply(predict_sample, axis=1)
samples[["Id", "Pawpularity", "predicted"]].rename(
    columns={"Pawpularity": "true_score", "predicted": "predicted_score"}
)

In [ ]:
# Visualise predictions vs ground truth on 8 sample images
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for ax, (_, row) in zip(axes, samples.iterrows()):
    img_path = DEBUG_IMG_DIR / (row["Id"] + ".jpg")
    img = Image.open(img_path).convert("RGB")
    ax.imshow(img)
    ax.set_title(
        f"True: {int(row['Pawpularity'])}  |  Pred: {row['predicted']:.1f}",
        fontsize=9
    )
    ax.axis("off")

plt.suptitle("Inference on Debug Samples — True vs Predicted Pawpularity", fontsize=11)
plt.tight_layout()
plt.show()

## 3. Validation RMSE on Debug Subset

Compute RMSE over all 100 debug images to sanity-check the loaded model.

In [ ]:
all_preds, all_targets = [], []

for _, row in df.iterrows():
    pred = predict_sample(row)
    all_preds.append(pred)
    all_targets.append(float(row["Pawpularity"]))

preds_t   = torch.tensor(all_preds).unsqueeze(1)
targets_t = torch.tensor(all_targets).unsqueeze(1)
debug_rmse = rmse(preds_t, targets_t)

print(f"Debug subset RMSE (100 images): {debug_rmse:.4f}")
print("(Full dataset val RMSE from training: 20.53)")

## 4. Grad-CAM Visualisation

Grad-CAM highlights which image regions the CNN focused on when predicting Pawpularity.  
Warm colours (red/yellow) = high importance. Cool colours (blue) = low importance.

In [ ]:
class GradCAM:
    """Minimal Grad-CAM using forward/backward hooks."""

    def __init__(self, model, target_layer):
        self.model       = model
        self.activations = None
        self.gradients   = None
        self._fh = target_layer.register_forward_hook(self._save_act)
        self._bh = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, m, inp, out):  self.activations = out.detach()
    def _save_grad(self, m, gi, go):   self.gradients   = go[0].detach()

    def __call__(self, img_t, tab_t):
        self.model.zero_grad()
        out = self.model(img_t, tab_t)
        out.backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=img_t.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    def remove(self):
        self._fh.remove(); self._bh.remove()


def overlay_heatmap(orig, heatmap, alpha=0.45):
    cmap    = plt.get_cmap("jet")
    colored = (cmap(heatmap)[:, :, :3] * 255).astype(np.uint8)
    return (alpha * colored + (1 - alpha) * orig).astype(np.uint8)


# Attach Grad-CAM to last EfficientNet block
grad_cam_model = model  # reuse already-loaded model (eval mode)
target_layer   = grad_cam_model.image_encoder.features[-1]
grad_cam       = GradCAM(grad_cam_model, target_layer)

print("GradCAM attached to:", target_layer.__class__.__name__)

In [ ]:
N = 8  # number of images to visualise
cam_samples = df.sample(N, random_state=0).reset_index(drop=True)

fig, axes = plt.subplots(2, N, figsize=(N * 2.5, 6))

for i, (_, row) in enumerate(cam_samples.iterrows()):
    img_path = DEBUG_IMG_DIR / (row["Id"] + ".jpg")
    img      = Image.open(img_path).convert("RGB").resize((224, 224), Image.LANCZOS)
    original = np.array(img)

    img_t = val_transform(img).unsqueeze(0).to(device)
    tab_t = torch.tensor(
        row[TABULAR_COLS].values.astype(np.float32)
    ).unsqueeze(0).to(device)

    heatmap   = grad_cam(img_t, tab_t)
    overlayed = overlay_heatmap(original, heatmap)

    with torch.no_grad():
        pred = model(img_t, tab_t).item()

    axes[0, i].imshow(original)
    axes[0, i].set_title(f"True: {int(row['Pawpularity'])}", fontsize=8)
    axes[0, i].axis("off")

    axes[1, i].imshow(overlayed)
    axes[1, i].set_title(f"Pred: {pred:.1f}", fontsize=8)
    axes[1, i].axis("off")

fig.suptitle(
    "Grad-CAM  |  Top: original  |  Bottom: heatmap overlay\n"
    "Warm = regions CNN weighted most heavily",
    fontsize=10
)
plt.tight_layout()
plt.show()

grad_cam.remove()

## 5. Reproduce Key Results Table

If you have trained all four checkpoints, this cell evaluates each on the debug subset and prints a summary table. (Full val RMSE requires the complete dataset — see README.)

In [ ]:
CHECKPOINTS = {
    "fusion_concat (baseline)": "checkpoints/fusion_concat.pt",
    "image_only":               "checkpoints/image_only.pt",
    "tabular_only":             "checkpoints/tabular_only.pt",
    "fusion_attention":         "checkpoints/fusion_attention.pt",
}

REPORTED_RMSE = {
    "fusion_concat (baseline)": 20.53,
    "image_only":               20.59,
    "tabular_only":             20.58,
    "fusion_attention":         20.60,
}

results = []

for name, ckpt_rel in CHECKPOINTS.items():
    ckpt_path = REPO_ROOT / ckpt_rel
    if not ckpt_path.exists():
        print(f"[SKIP] {name}: checkpoint not found at {ckpt_path}")
        results.append({"Model": name, "Debug RMSE": "—", "Reported Val RMSE": REPORTED_RMSE[name]})
        continue

    ck   = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg_ = ck["cfg"]
    m = PawpularityModel(
        fusion_mode=cfg_.get("fusion_mode", "fusion_concat"),
        freeze_backbone=cfg_["freeze_backbone"],
        tabular_input_dim=cfg_["tabular_input_dim"],
        tabular_hidden_dim=cfg_["tabular_hidden_dim"],
        tabular_output_dim=cfg_["tabular_output_dim"],
        fusion_hidden_dim=cfg_["fusion_hidden_dim"],
        dropout=cfg_["dropout"],
    )
    state_ = {k.replace("fusion_head.", "head."): v for k, v in ck["model_state_dict"].items()}
    m.load_state_dict(state_, strict=False)
    m.to(device).eval()

    preds_, targets_ = [], []
    for _, row in df.iterrows():
        img_path = DEBUG_IMG_DIR / (row["Id"] + ".jpg")
        img      = Image.open(img_path).convert("RGB")
        img_t    = val_transform(img).unsqueeze(0).to(device)
        tab_t    = torch.tensor(
            row[TABULAR_COLS].values.astype(np.float32)
        ).unsqueeze(0).to(device)
        with torch.no_grad():
            preds_.append(m(img_t, tab_t).item())
        targets_.append(float(row["Pawpularity"]))

    r = rmse(torch.tensor(preds_).unsqueeze(1), torch.tensor(targets_).unsqueeze(1))
    results.append({"Model": name, "Debug RMSE": f"{r:.2f}", "Reported Val RMSE": REPORTED_RMSE[name]})
    print(f"{name:35s}  debug RMSE={r:.4f}  reported={REPORTED_RMSE[name]}")

print()
pd.DataFrame(results)

### Notes
- **Debug RMSE** is computed on the 100-image debug subset and will differ from the reported val RMSE (which uses the full ~9,900-image dataset with an 80/20 train/val split).
- To reproduce the exact reported val RMSE, download the full dataset and re-run training as described in the README.
- All experiments used `seed=42`, AdamW optimiser, cosine annealing LR schedule, 25 epochs, batch size 32.